In [ ]:
from pathlib import Path
import pandas as pd

from ultralytics import YOLO

BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

YOLO_REY_DIR = BASE_DIR / "yolo" / "rey_bbox"
YAML_PATH = YOLO_REY_DIR / "rey_bbox.yaml"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"

print("YAML existe:", YAML_PATH.exists())
print("YAML:", YAML_PATH)

In [ ]:
import ultralytics
print("Ultralytics:", ultralytics.__version__)

In [ ]:
model = YOLO("yolo11n.pt")

In [ ]:
results = model.train(
    data=str(YAML_PATH),
    epochs=30,
    imgsz=640,
    batch=8,
    device="cpu",
    project=str(BASE_DIR / "outputs" / "yolo_runs"),
    name="rey_bbox_yolo11n",
    exist_ok=True
)

In [ ]:
best_model_path = BASE_DIR / "outputs" / "yolo_runs" / "rey_bbox_yolo11n" / "weights" / "best.pt"

print("Best model existe:", best_model_path.exists())
print(best_model_path)

In [ ]:
best_model = YOLO(str(best_model_path))

metrics = best_model.val(
    data=str(YAML_PATH),
    split="test",
    imgsz=640,
    device="cpu"
)

In [ ]:
test_images_dir = YOLO_REY_DIR / "images" / "test"

test_images = list(test_images_dir.glob("*.jpg")) + list(test_images_dir.glob("*.png"))

print("Imágenes test:", len(test_images))
print(test_images[:3])

In [ ]:
pred_results = best_model.predict(
    source=[str(p) for p in test_images[:6]],
    imgsz=640,
    conf=0.25,
    device="cpu",
    save=True,
    project=str(BASE_DIR / "outputs" / "yolo_predictions"),
    name="rey_bbox_test_examples",
    exist_ok=True
)

In [ ]:
import pandas as pd

tabla_yolo_bbox = pd.DataFrame({
    "modelo": ["YOLO11n"],
    "precision": [0.992],
    "recall": [0.992],
    "mAP50": [0.995],
    "mAP50_95": [0.925],
    "split": ["test"]
})

tabla_path = TABLES_DIR / "rey_bbox_yolo_resultados_test.csv"
tabla_yolo_bbox.to_csv(tabla_path, index=False)

print("Tabla guardada en:", tabla_path)
display(tabla_yolo_bbox)

In [ ]:
tabla_comparativa_bbox = pd.DataFrame({
    "modelo": ["CNN Keras", "YOLO11n"],
    "metrica_principal": ["IoU media", "mAP50-95"],
    "valor_principal": [0.7286, 0.925],
    "extra_1": ["IoU>0.5 = 0.9398", "mAP50 = 0.995"],
    "extra_2": ["IoU>0.7 = 0.5714", "P=0.992, R=0.992"]
})

tabla_path = TABLES_DIR / "rey_bbox_comparativa_keras_yolo.csv"
tabla_comparativa_bbox.to_csv(tabla_path, index=False)

display(tabla_comparativa_bbox)
print("Tabla guardada en:", tabla_path)